# Module 5: Capstone Deal Agent

**Goal:** Build a multi-agent system that scans for deals, estimates their fair value, and identifies opportunities.

## 1. Setup & Dependencies
We use the refactored agents in `reference_code`.

In [1]:
import sys
import os
from dotenv import load_dotenv
from reference_code.deal_agents import ScannerAgent, PricerAgent
from reference_code.deal_models import Opportunity

load_dotenv(override=True)
print("Agents loaded.")

Agents loaded.


## 2. The Deal Agent Workflow
We orchestrate the `ScannerAgent` (RSS -> Structured Data) and `PricerAgent` (LLM Estimation).

In [2]:
def run_deal_agent_pipeline():
    # 1. Initialize Agents
    scanner = ScannerAgent()
    pricer = PricerAgent()
    memory = [] # Keep track of URLs to avoid dupes in a long-running app

    # 2. Scan for Deals
    print("\n--- 📡 Phase 1: Scanning Deals ---")
    selection = scanner.scan(memory=memory)
    
    if not selection or not selection.deals:
        print("No valid deals found.")
        return

    print(f"Found {len(selection.deals)} candidate deals.")

    # 3. Price & Evaluate
    print("\n--- 🏷️ Phase 2: Pricing & Evaluation ---")
    opportunities = []
    
    for deal in selection.deals:
        estimated_value = pricer.estimate_value(deal)
        discount = estimated_value - deal.price
        
        if discount > 0:
            opp = Opportunity(
                deal=deal,
                estimate=estimated_value,
                discount=discount
            )
            opportunities.append(opp)
            print(f"✅ Opportunity! {deal.title[:40]}... | Price: ${deal.price} | Value: ${estimated_value} | Profit: ${discount}")
        else:
            print(f"❌ Pass.        {deal.title[:40]}... | Price: ${deal.price} | Value: ${estimated_value}")

    # 4. Report
    print("\n--- 🏆 Phase 3: Final Report ---")
    if opportunities:
        best = max(opportunities, key=lambda x: x.discount)
        print(f"Best Deal: {best.deal.title}")
        print(f"Potential Profit: ${best.discount:.2f}")
        print(f"Link: {best.deal.url}")
    else:
        print("No profitable opportunities found in this batch.")

if __name__ == "__main__":
    run_deal_agent_pipeline()

[Scanner Agent] Initializing

--- 📡 Phase 1: Scanning Deals ---
[Scanner Agent] Fetching deals from RSS feeds...


100%|██████████| 3/3 [00:55<00:00, 18.42s/it]


[Scanner Agent] Found 15 new deals.
[Scanner Agent] Analyzing deals with LLM...
Found 5 candidate deals.

--- 🏷️ Phase 2: Pricing & Evaluation ---
[Pricer Agent] Estimating value for: These M01 Pro Smart Glasses are equipped with an 8...
[Pricer Agent] Estimated: $250.0
✅ Opportunity! M01 Pro 800W HD Smart Glasses for $27 + ... | Price: $27.0 | Value: $250.0 | Profit: $223.0
[Pricer Agent] Estimating value for: The Sennheiser HD 450BT wireless headphones delive...
[Pricer Agent] Estimated: $150.0
✅ Opportunity! Sennheiser HD 450BT Wireless Bluetooth H... | Price: $75.0 | Value: $150.0 | Profit: $75.0
[Pricer Agent] Estimating value for: Experience an immersive 3D audio experience with t...
[Pricer Agent] Estimated: $499.0
❌ Pass.        Sennheiser Ambeo Soundbar Mini with 3D A... | Price: $500.0 | Value: $499.0
[Pricer Agent] Estimating value for: The ASUS Vivobook Go is powered by a 13th Generati...
[Pricer Agent] Estimated: $550.0
✅ Opportunity! ASUS Vivobook Go 15.6" 13th-Gen i3 Lap